In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:


# 1. LOAD DATAHealthTrends = pd.read_csv('HealthTrends.csv')
#HealthTrends.columns
HealthTrends

In [ ]:
# 2. FEATURE SELECTION
features = [
    'Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 
    'Torque_Nm', 'Tool_wear_min', 'Temp_Delta', 'Power_Output_Watts', 
    'Cumulative_Strain', 'Rolling_Temp_Avg', 'Torque_Instant_Change'
]

# Defining Machine_failure
X = HealthTrends[features]
y = HealthTrends['Machine_failure']

In [ ]:
# 3. TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. INITIALIZE & TRAIN XGBOOST
# Used'scale_pos_weight' because machine failures are rare (Imbalanced Data)
model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=10, # Helps the model find the rare 1% of failures
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

In [ ]:
# 5. PREDICTION & RISK COSTING
y_pred = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1] # Probability of failure
y_probs

In [ ]:
# 6. FEATURE IMPORTANCE: Which sensor is the "Penalty Killer"?
plt.figure(figsize=(10,6))
xgb.plot_importance(model, importance_type='gain', max_num_features=10)
plt.title('Operational Risk Drivers (Feature Importance)')
plt.show()

In [ ]:
# 7. BUSINESS LOGIC: Mapping Risk to Dollars
# Simulate a "Financial Risk" based on $5,000/hr downtime logic
test_results = X_test.copy()
test_results['Failure_Probability'] = y_probs
test_results['Financial_Exposure_USD'] = test_results['Failure_Probability'] * 5000 

print(test_results[['Failure_Probability', 'Financial_Exposure_USD']].head(10))